# API GOOGLE DRIVE

resumo oficial: https://developers.google.com/workspace/drive/api/reference/rest/v3?utm_source=chatgpt.com&hl=pt-br

documentação python: https://developers.google.com/workspace/drive/api/quickstart/python?utm_source=chatgpt.com&hl=pt-br

In [1]:
# pip install --upgrade google-api-python-client \
#     google-auth-httplib2 \
#     google-auth-oauthlib

In [2]:

import os.path

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

SCOPES = [
    "https://www.googleapis.com/auth/drive"
]


In [6]:

def conectar_drive():
    creds = None

    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file(
            "token.json", SCOPES
        )

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                "credentials.json", SCOPES
            )
            creds = flow.run_local_server(port=0)

        with open("token.json", "w") as token:
            token.write(creds.to_json())

    return build("drive", "v3", credentials=creds)


drive = conectar_drive()


Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=1040055329703-lupeqmn9552h97ui9c38gaucc9r95ghf.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A48973%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive&state=G7IIQGkDwcyPTnEgy8zdwXQEtNh28U&code_challenge=F9AXAJRec8mYuAz6VF-4c4o6F3EkjqWznJJpJsdwGEs&code_challenge_method=S256&access_type=offline


In [7]:
drive

# Listar Arquivos

In [16]:
resultado = drive.files().list(
    pageSize=200,
    orderBy="name",
    fields="files(id, name, mimeType, webViewLink)"
).execute()

for arquivo in resultado["files"]:
    print(
        arquivo["name"], "|",
        arquivo["id"],
        # arquivo["mimeType"]
    )

  Frequência período da tarde - 2026 | 1JkvgDI0-yz8CsHtkdePY47IbS-9FG0dVtRUbyGTl9xw
 Frequência período da manhã - 2026 | 1vk70CabH_dP-U_EBxoeHpe4zuY40q6kyTTIFcmiwc84
 WILCK | 1ZgxJP2AhN6oaCOFwyEPYxI8GkDHkl3DT
.DS_Store | 16aaCdJIC-5aFmK0aV7IbGsDih6Qp5u2r
.env | 1yiGylUkCRfkMloFQpZ947c7WY2DTFnbB
.idea | 1VOGxWZ6XoL_EsWTNtoWokNFNg8V7jdq3
.ipynb_checkpoints | 1LE6dtV19LX1k2_EdNELPJ8zEXIvtk1Hu
.ipynb_checkpoints | 1eMXifjlDA00E0_bZ_jMatFKZQv7pyHmS
.ipynb_checkpoints | 1HQDyUq4VyIomP_znA_GOeuvCZP26vEwR
.ipynb_checkpoints | 1zTjn_syaxv6WX5f4BvaUQ7l-Tp7x_Ol7
.ipynb_checkpoints | 1vMJPn4kvGdPMCCAgCuQ3JOBdF7y0UlYf
.ipynb_checkpoints | 1UaRwDJoApy_UQ3HbF7ghXfOWq4pqWFU3
.ipynb_checkpoints | 1MrabrvVihGsN6N_uK_551ynaJniLns_X
.ipynb_checkpoints | 14p5hajkYQXSwZfJZ_-iBrYU8sksCKB-H
.~lock.1 - Inteligência Artificial.pptx# | 1fljZG_bv3ZgXsP0JOUQDHHGIKTsNzX1o
.~lock.2 - Aprendizado de Máquina.pptx# | 1w8hHH_pZh_gWWRUYXY4M0rP4_MG4Th_n
00 - Atividades em Geral em Python | 1kUoRhsYZiYAQ77zC9H9slqlAQxhsro

# Criar uma pasta

In [12]:
pasta = drive.files().create(
    body={
        "name": "Projetos Python",
        "mimeType": "application/vnd.google-apps.folder"
    },
    fields="id, name"
).execute()

print(pasta)

{'id': '1ni6O3JLAFRHSszaSyavDQbLdN0t6o50k', 'name': 'Projetos Python'}


## Criar uma subpasta

In [19]:
subpasta = drive.files().create(
    body={
        "name": "Relatorios",
        "mimeType": "application/vnd.google-apps.folder",
        "parents": [pasta["id"]]
    },
    fields="id, name"
).execute()

# Fazendo Upload de um arquivo

In [15]:
from googleapiclient.http import MediaFileUpload

arquivo = drive.files().create(
    body={
        "name": "teste.txt",
        "parents": [pasta["id"]]
    },
    media_body=MediaFileUpload(
        "teste.txt",
        mimetype="text/txt"
    ),
    fields="id, name, webViewLink"
).execute()

print(arquivo)

{'id': '15ZhtvtbpXwgYBFu7sFl5t-b9r53UsOv6', 'name': 'teste.txt', 'webViewLink': 'https://drive.google.com/file/d/15ZhtvtbpXwgYBFu7sFl5t-b9r53UsOv6/view?usp=drivesdk'}


# Baixar um arquivo

In [18]:
from googleapiclient.http import MediaIoBaseDownload
import io

file_id = "13UU3phWW7MzhFPcDxRrDIpkOWDZK2lmN"
request = drive.files().get_media(fileId=file_id)

with open("arquivo_baixado.jpg", "wb") as destino:
    downloader = MediaIoBaseDownload(destino, request)

    concluido = False
    while not concluido:
        status, concluido = downloader.next_chunk()
        print(f"{int(status.progress() * 100)}%")

100%


# Renomear um arquivo

In [ ]:
drive.files().update(
    fileId="ID_DO_ARQUIVO",
    body={"name": "relatorio_final.pdf"}
).execute()

# Mover um arquivo para outra pasta

In [ ]:
file_id = "ID_DO_ARQUIVO"
nova_pasta_id = "ID_DA_NOVA_PASTA"

arquivo = drive.files().get(
    fileId=file_id,
    fields="parents"
).execute()

pais_atuais = ",".join(arquivo.get("parents", []))

drive.files().update(
    fileId=file_id,
    addParents=nova_pasta_id,
    removeParents=pais_atuais
).execute()

# Excluir um arquivo

In [ ]:
drive.files().delete(
    fileId="ID_DO_ARQUIVO"
).execute()

# Resumo das funções

| Operação | Método principal |
|---|---|
| Listar arquivos | `files().list()` |
| Consultar metadados | `files().get()` |
| Criar pasta ou arquivo | `files().create()` |
| Enviar arquivo | `files().create()` + `MediaFileUpload` |
| Baixar arquivo | `files().get_media()` |
| Renomear ou atualizar | `files().update()` |
| Mover entre pastas | `files().update()` |
| Excluir permanentemente | `files().delete()` |
| Compartilhar | `permissions().create()` |